<a href="https://colab.research.google.com/github/Sunil-Paliwal/ai-enabled-drug-discovery-and-synthesis/blob/main/04_aizynthfinder_retrosynthesis/aizynthfinder_retrosynthesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AiZynthFinder Retrosynthesis in Google Colab

<div style="font-size:13px; color:#333; border-left:3px solid #2c5c40;
            padding:8px 16px; margin:12px 0; background:#f7f9f7;">
<b>Author:</b> Dr. Sunil Paliwal<br>
Department of Chemistry and Chemical Biology<br>
Stevens Institute of Technology<br><br>
<b>Module:</b> 04 of <a href="https://doi.org/10.5281/zenodo.24232866">AI-Enabled
Drug Discovery and Synthesis</a> &nbsp;&middot;&nbsp;
<b>Version:</b> 1.0.1 &nbsp;&middot;&nbsp; <b>Licence:</b> MIT<br>
<b>Status:</b> Developed for teaching and research in drug discovery and medicinal
chemistry.<br>
<b>Cite as:</b> Paliwal, S. <i>AI-Enabled Drug Discovery and Synthesis.</i> Zenodo, 2026.
doi.org/10.5281/zenodo.24232866<br>
<span style="font-size:12px; color:#666;">Working from a saved copy? The link above
always has the current version.</span>
</div>

---

**Can this molecule actually be made, by what synthetic route, and through which reactions?**

This notebook runs [AiZynthFinder](https://github.com/MolecularAI/aizynthfinder)
(Genheden *et al.*, *J. Cheminform.* **2020**, *12*, 70) to work backwards from a
target structure to purchasable starting materials, using a neural-network
expansion policy to propose disconnections and Monte Carlo tree search to
assemble them into routes.

**What you get back**

* A scheme for each route, with the **name of each reaction** written beside it.
* For every starting material, **which catalogue it was found in** &mdash; not just
  whether it is "in stock".
* A **PDF report** of all routes with their step and starting-material tables.
* A **ChemDraw bundle**: MDL `.rxn` files you can open directly, reaction SMILES
  for Paste Special, and an SDF of every compound.

**How to use it:** run the cells in order. Cells 1 to 3 set the session up and
only need running once; after that, type a SMILES into Cell 4 and search as
often as you like.

| | what it does | how long |
|---|---|---|
| **Cell 1** | installs the search engine and the reaction namer | ~2 min |
| **Cell 2** | downloads the models and the catalogue (1.3 GB) | ~3&ndash;4 min |
| **Cell 3** | writes the support code | instant |
| **Cell 4** | one search | ~2&ndash;3 min |

So about eight minutes from a cold session to your first set of routes, and two
to three minutes for each search after that. If the Colab runtime disconnects,
Cells 1 to 3 have to run again &mdash; nothing is kept between sessions.

---

**These routes are predictions, not recipes.** The models are trained on
reactions extracted from US patents, which are not a complete or unbiased
record of organic chemistry. A route that scores well may still use a step
that does not work on your substrate, and reaction names group transformations
by the bond formed rather than by mechanism. Nothing here has been checked
against the literature. Treat every route as a hypothesis to verify, and
confirm precedent and conditions before attempting any of it.

## Cell 1 &mdash; Install

AiZynthFinder supports Python 3.9&ndash;3.11, and Colab's default Python is newer than
that, so it goes into its own Python 3.11 environment. Everything else &mdash;
reaction naming, drawing, the PDF and the ChemDraw export &mdash; runs in Colab's own
Python, which already has the PyTorch that the atom-mapper needs.

**Expect two to three minutes.** It reports how long it took when it finishes.
Cells 1 and 2 only need running once per session; after that you can search as
often as you like.

In [ ]:
# ---- Cell 1: install -------------------------------------------------------
import logging, os, shutil, subprocess, sys, time

t0 = time.time()
os.environ.pop("UV_SYSTEM_PYTHON", None)


def run_quiet(cmd, what):
    """Install quietly; show the output only if something actually broke.

    pip prints a red dependency-resolver ERROR whenever a new package
    disagrees with something Colab preinstalled. Here that is gradio and
    diffusers objecting to the huggingface-hub pin that the atom-mapper
    brings with it - neither is used by this notebook, so the complaint is
    noise, but it reads like a failure. The import checks further down are
    what actually decide whether the install worked.
    """
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
    except FileNotFoundError:
        raise RuntimeError(f"{what} failed: '{cmd[0]}' was not found. "
                           "Re-run this cell; if it persists, use "
                           "Runtime > Restart session and run it again.")
    if result.returncode != 0:
        print(f"  {what} FAILED:")
        print((result.stderr or result.stdout)[-2500:])
        raise RuntimeError(f"{what} failed - see the output above.")
    return result


run_quiet([sys.executable, "-m", "pip", "install", "-q", "uv"], "uv")

# Resolve the binary rather than trusting PATH: it was installed moments ago.
UV = shutil.which("uv") or os.path.join(os.path.dirname(sys.executable), "uv")
run_quiet([UV, "venv", "--clear", "--python", "3.11", "/content/azf_env"],
          "Python 3.11 environment")

# NOTE on "pyarrow<26":
# pyarrow arrives as an unpinned transitive dependency
#   aizynthfinder -> reaction-utils -> swifter -> dask[dataframe] -> pyarrow>=16.0
# pyarrow 26.0.0 dropped NumPy 1.x, but aizynthfinder pins numpy<2.0.0. So the
# newest pyarrow installs without any resolver error and then raises
#   ImportError: pyarrow requires NumPy 2.0 or newer, found 1.26.4
# on the first import. Capping it below 26 keeps it compatible with numpy 1.26.
print("Installing AiZynthFinder into its own Python 3.11 environment...")
run_quiet([UV, "pip", "install", "-q", "--python", "/content/azf_env/bin/python",
           "aizynthfinder==4.4.1", "pyarrow<26"], "AiZynthFinder")

# Main kernel: reaction naming, PDF. rdkit and PIL are already in Colab.
print("Installing reaction-naming and reporting tools...")
run_quiet([sys.executable, "-m", "pip", "install", "-q", "rxn-insight", "reportlab"],
          "reaction-naming tools")

# Surface environment problems here rather than as an opaque "Search failed"
# message from the search subprocess later.
check = subprocess.run(
    ["/content/azf_env/bin/python", "-c",
     "import numpy, pyarrow; "
     "from aizynthfinder.aizynthfinder import AiZynthFinder; "
     "print(f'AiZynthFinder OK (numpy {numpy.__version__}, pyarrow {pyarrow.__version__})')"],
    capture_output=True, text=True)
print(check.stdout.strip() or check.stderr.strip()[-2000:])
if check.returncode != 0:
    raise RuntimeError("Environment check failed - see the error above.")

# Warm up the atom-mapper now. Its model loads from disk on first use, which
# takes up to a minute cold; paying it here keeps it out of your first search.
print("Warming up the reaction namer (one-off, up to a minute)...")
try:
    import warnings; warnings.filterwarnings("ignore")
    # torch logs a Triton notice on CPU-only runtimes; it has no bearing on
    # anything here and only makes the output look alarming.
    logging.getLogger("torch").setLevel(logging.ERROR)
    from rxn_insight.reaction import Reaction
    name = Reaction("CC(=O)O.NC>>CC(=O)NC").get_reaction_info().get("NAME")
    print(f"Reaction namer OK (test reaction named: {name})")
except Exception as exc:
    print(f"Reaction namer unavailable ({type(exc).__name__}) - "
          "routes will still work, but steps will be unnamed.")

print(f"\nCell 1 complete in {time.time()-t0:.0f}s.")

## Cell 2 &mdash; Models and catalogues

Downloads the public USPTO models and the ZINC stock collection.

**Expect three to five minutes**, most of it on a single 1.3 GB file with no
progress indicator, so a long silence is normal. The cell reports how long it
took when it finishes.

The models are split across two public hosts and one of them, figshare, refuses
a share of requests. When that happens the cell says so and fetches the missing
files from a mirror instead. It is routine and needs nothing from you.

**About the catalogue.** ZINC ships with AiZynthFinder and is used by default.
It tells you a compound was purchasable *somewhere*, but the bundled snapshot
dates from 2020 and carries no supplier or delivery information, so treat a
green frame as "this was buyable five years ago" rather than "you can order
this today". The notebook prints the catalogue name against every starting
material for that reason.

**Adding your own catalogue.** If you have access to a supplier catalogue, you
can convert it with `Build_Stock_File.ipynb` and list it in `EXTRA_STOCKS`
above. The notebook then reports which catalogue each starting material came
from, so a current entry is distinguishable from a 2020 one. Download any such
catalogue under your own account and on the supplier's own terms; nothing of
the kind is distributed with these notebooks.

Worth knowing when reading the results either way: supplier catalogues of
*novel* building blocks often omit commodity reagents. Piperidine and benzyl
bromide may be missing from one although any lab can buy them in a day, while
ZINC has them. Catalogues complement each other, which is why all of them are
reported rather than reduced to a single in-stock flag.

In [ ]:
# ---- Cell 2: models and catalogues -----------------------------------------
# Optional extra catalogues, on top of ZINC. Leave this empty to use ZINC
# alone. To add a supplier catalogue you have downloaded under your own
# account, convert it with Build_Stock_File.ipynb and add it here as
# "label": "/path/to/file.hdf5" -- the label is what gets printed next to
# each starting material, so name it something you will recognise:
#
#     EXTRA_STOCKS = {"my supplier, Oct 2026": "/content/my_stock.hdf5"}
#
EXTRA_STOCKS = {}

import os, subprocess, time, yaml

PY = "/content/azf_env/bin/python"
expected = ["uspto_model.onnx", "uspto_templates.csv.gz",
            "uspto_ringbreaker_model.onnx", "uspto_ringbreaker_templates.csv.gz",
            "uspto_filter_model.onnx", "zinc_stock.hdf5"]

def missing_files():
    return [f for f in expected
            if not os.path.exists(f"/content/{f}") or os.path.getsize(f"/content/{f}") == 0]

t0 = time.time()

# download_public_data re-fetches everything unconditionally, all 1.3 GB of it,
# with no check for what is already on disk. Since changing the catalogue list
# above means re-running this cell, skip the download when the files are
# already here.
if missing_files() or not os.path.exists("/content/config.yml"):
    print("Downloading models and the ZINC stock. Three to five minutes; the")
    print("stock file alone is 1.3 GB, so there are long quiet stretches.")

    # Run this quietly rather than streaming it. AiZynthFinder's downloader
    # prints "Download failed with message 403 Client Error: Forbidden" when
    # figshare declines a request, which happens routinely and reads like a
    # fatal error even though the mirror below handles it.
    subprocess.run(["/content/azf_env/bin/download_public_data", "/content"],
                   capture_output=True, text=True)
else:
    print("Models and ZINC stock already downloaded - skipping.")

# The standard download stops at the first failure (typically a 403 from
# figshare.com, which hosts the ZINC stock). Anything missing comes from a
# backup copy of the same public files on Zenodo.
missing = missing_files()
if missing:
    print(f"  one host was unavailable; fetching {len(missing)} file(s) from the "
          "Zenodo mirror instead (this is routine)...")
    for f in missing:
        dest = f"/content/{f}"
        r = subprocess.run(["wget", "-q", "-O", dest,
                            f"https://zenodo.org/records/11430881/files/{f}?download=1"])
        if r.returncode != 0:
            if os.path.exists(dest):
                os.remove(dest)
            raise RuntimeError(f"Could not download {f} from the Zenodo backup either. "
                               "Wait a few minutes and re-run this cell.")
    subprocess.run([PY, "-c",
                    "import sys; "
                    "from aizynthfinder.tools import download_public_data as d; "
                    "sys.argv = ['download_public_data', '/content']; "
                    "d._download_file = lambda url, filename: None; "
                    "d.main()"], check=True)

still_missing = missing_files() + ([] if os.path.exists("/content/config.yml") else ["config.yml"])
if still_missing:
    raise RuntimeError(f"Missing after download: {still_missing}. Re-run this cell; "
                       "if it keeps failing the public model hosts may be unreachable.")

# Add any extra catalogues that are actually present to config.yml.
config = yaml.safe_load(open("/content/config.yml"))
config.setdefault("stock", {})
STOCKS = ["zinc"]
for key, path in EXTRA_STOCKS.items():
    if path and os.path.exists(path):
        config["stock"][key] = path
        STOCKS.append(key)
        print(f"  added catalogue '{key}': {os.path.basename(path)} "
              f"({os.path.getsize(path)/1e6:,.0f} MB)")
    elif path:
        print(f"  '{key}' not found at {path} - skipping")
yaml.safe_dump(config, open("/content/config.yml", "w"), sort_keys=False)

# Make sure the models and every selected stock actually load, so a bad
# download shows up here rather than mid-search.
check = subprocess.run(
    [PY, "-c",
     "from aizynthfinder.aizynthfinder import AiZynthFinder; "
     f"f = AiZynthFinder(configfile='/content/config.yml'); f.stock.select({STOCKS!r}); "
     "print('Models and stock load OK')"],
    capture_output=True, text=True, cwd="/content")
if check.returncode != 0:
    print(check.stderr.strip()[-2000:])
    raise RuntimeError("The downloaded files did not load - see the error above. "
                       "Delete them with: "
                       "!rm -f /content/*.onnx /content/*.hdf5 /content/*.csv.gz  "
                       "then re-run this cell.")
print(check.stdout.strip())
print(f"\nCatalogues in use: {', '.join(STOCKS)}")
print(f"Cell 2 complete in {time.time()-t0:.0f}s.")

## Cell 3 &mdash; Support code

Writes the module that names reactions, draws the schemes, builds the PDF and
writes the ChemDraw files. You do not need to read it; just run it.

In [ ]:
%%writefile /content/azf_post.py
"""
Post-processing for AiZynthFinder routes: reaction names, labelled schemes,
PDF reports and ChemDraw-ready exports.

Runs in the main Colab kernel, not the Python 3.11 search environment, so that
the transformer atom-mapper can reuse the torch that Colab already provides.

Author: Dr. Sunil Paliwal
Department of Chemistry and Chemical Biology
Stevens Institute of Technology

------------------------------------------------------------------------------
Portions of this file (molecule_to_image, molecules_to_images, crop_image,
draw_rounded_rectangle and the route layout arithmetic) are adapted from
reaction-utils (rxnutils.routes.image), copyright AstraZeneca, licensed under
the Apache License 2.0. Modifications: the route factory now captures reaction
nodes and draws their names, and crop_image uses a bounding-box difference
instead of a per-pixel scan.
------------------------------------------------------------------------------
"""

import io
import os
import re

from PIL import Image, ImageChops, ImageDraw, ImageFont
from rdkit import Chem
from rdkit.Chem import AllChem, Draw
from rdkit.Chem import rdChemReactions as rdRxn


# ==========================================================================
# Drawing helpers (adapted from rxnutils, Apache 2.0)
# ==========================================================================
def crop_image(img, margin=20):
    """Trim white space, then re-pad.

    The original walks every pixel in Python -- 90,000 getpixel calls for a
    300x300 molecule, repeated for every structure in every route. Differencing
    against a white background and asking PIL for the bounding box is the same
    answer from C.
    """
    rgb = img.convert("RGB")
    diff = ImageChops.difference(rgb, Image.new("RGB", rgb.size, (255, 255, 255)))
    bbox = diff.getbbox()
    cropped = img.crop(bbox) if bbox else img
    out = Image.new(img.mode,
                    (cropped.width + 2 * margin, cropped.height + 2 * margin),
                    color="white")
    out.paste(cropped, (margin + 1, margin + 1))
    return out


def draw_rounded_rectangle(img, color, arc_size=20):
    """Draw a rounded frame around an image."""
    x0, y0, x1, y1 = img.getbbox()
    x1 -= 1
    y1 -= 1
    copy = img.copy()
    draw = ImageDraw.Draw(copy)
    half = arc_size // 2
    draw.arc((x0, y0, arc_size, arc_size), start=180, end=270, fill=color)
    draw.arc((x1 - arc_size, y0, x1, arc_size), start=270, end=0, fill=color)
    draw.arc((x1 - arc_size, y1 - arc_size, x1, y1), start=0, end=90, fill=color)
    draw.arc((x0, y1 - arc_size, arc_size, y1), start=90, end=180, fill=color)
    draw.line((x0 + half, y0, x1 - half, y0), fill=color)
    draw.line((x1, half, x1, y1 - half), fill=color)
    draw.line((half, y1, x1 - half, y1), fill=color)
    draw.line((x0, half, x0, y1 - half), fill=color)
    return copy


def molecules_to_images(mols, frame_colors, size=300, draw_kwargs=None):
    """Draw molecules at a consistent scale, each in a coloured frame."""
    draw_kwargs = draw_kwargs or {}
    grid = Draw.MolsToGridImage(mols, molsPerRow=len(mols),
                                subImgSize=(size, size), **draw_kwargs)
    if not hasattr(grid, "crop"):          # not a PIL image (SVG/IPython)
        grid = Image.open(io.BytesIO(grid.data))
    images = []
    for idx, frame_color in enumerate(frame_colors):
        tile = grid.crop((size * idx, 0, size * (idx + 1), size))
        images.append(draw_rounded_rectangle(crop_image(tile), frame_color))
    return images


# ==========================================================================
# Fonts
# ==========================================================================
_FONT_PATHS = [
    "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
    "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf",
    "/usr/share/fonts/truetype/freefont/FreeSans.ttf",
]


def _font(size=15, bold=False):
    paths = _FONT_PATHS[:]
    if bold:
        paths = [p for p in paths if "Bold" in p] + paths
    for path in paths:
        if os.path.exists(path):
            try:
                return ImageFont.truetype(path, size)
            except Exception:
                pass
    try:
        return ImageFont.load_default(size=size)
    except TypeError:
        return ImageFont.load_default()


def _text_width(font, text):
    try:
        return font.getlength(text)
    except AttributeError:
        return font.getsize(text)[0]


def _wrap_to_px(text, font, max_px, max_lines=3):
    """Greedy word wrap measured in pixels, not characters.

    Wrapping by character count is what makes labels overrun the gap between
    two structures, because the gap is a pixel width and a character is not.
    """
    words, lines, cur = text.split(), [], ""
    for word in words:
        trial = f"{cur} {word}".strip()
        if cur and _text_width(font, trial) > max_px:
            lines.append(cur)
            cur = word
            if len(lines) == max_lines:
                break
        else:
            cur = trial
    if cur and len(lines) < max_lines:
        lines.append(cur)
    if lines and len(lines) == max_lines and len(" ".join(lines)) < len(text):
        last = lines[-1]
        while last and _text_width(font, last + "…") > max_px:
            last = last[:-1]
        lines[-1] = last.rstrip() + "…"
    out = []
    for line in lines:
        while line and _text_width(font, line) > max_px:
            line = line[:-1]
        out.append(line)
    return [ln for ln in out if ln]


# ==========================================================================
# Reaction naming
# ==========================================================================
# The classification column in AiZynthFinder's public USPTO template file is
# present but empty in every one of its 42,554 rows, so the template library
# cannot name anything. Names instead come from Rxn-INSIGHT, which classifies
# a reaction from its atom-mapped reaction centre.
#
#   Dobbelaere et al., "Rxn-INSIGHT: fast chemical reaction analysis using
#   bond-electron matrices", J. Cheminform. 2024, 16, 37.
#
# Caveat worth repeating to students: these names group reactions by the bond
# formed, not by mechanism. An activated aryl fluoride reacting with an amine
# comes back under the N-arylation heading even though it is an SNAr and needs
# no metal.

_UNNAMED = {"", "otherreaction", "other", "unknown", "nan", "none"}
_namer_warned = [False]


def name_reaction(reaction_smiles):
    """Return (name, reaction_class) for a forward reaction SMILES.

    Returns (None, None) when the reaction cannot be named or Rxn-INSIGHT is
    unavailable -- never raises, since one odd step should not lose a route.
    """
    if not reaction_smiles or ">>" not in reaction_smiles:
        return None, None
    try:
        from rxn_insight.reaction import Reaction
    except Exception:
        if not _namer_warned[0]:
            print("  (Rxn-INSIGHT not installed - steps will be unnamed)")
            _namer_warned[0] = True
        return None, None
    try:
        info = Reaction(reaction_smiles).get_reaction_info()
    except Exception:
        return None, None
    name = _tidy_name(str(info.get("NAME") or ""))
    klass = _tidy_name(str(info.get("CLASS") or ""))
    if not name or name.lower() in _UNNAMED:
        name = None
    if not klass or klass.lower() in _UNNAMED:
        klass = None
    return name, klass


def _tidy_name(text):
    """Strip the library's internal suffixes from a reaction name.

    Rxn-INSIGHT returns entries such as
      "Acylation of Nitrogen Nucleophiles by Acyl Halides and Analogs_OS"
    where the trailing _OS marks the sub-library the rule came from. It means
    nothing to a chemist reading a scheme.
    """
    text = re.sub(r"_(OS|US|RX|FG)\b", "", text.strip())
    return re.sub(r"\s+", " ", text).strip(" _-")


# One limit, used by both the scheme drawing and the CDXML export. Two
# different limits meant a step could read "Ester saponification (alkyl
# deprotection)" in the notebook and "Deprotection" in ChemDraw -- the same
# reaction under two names, which is worse than either name alone.
_NAME_LIMIT = 46


def display_name(step):
    """The reaction name as shown everywhere: scheme, CDXML and PDF table.

    A very long name truncates to jargon ("Acylation of Nitrogen Nucleophiles
    by Acyl/Thioacyl/Carbam...") and says less than the one-word class does,
    so an over-long name gives way to its class. The full name is always kept
    in the PDF's step table.
    """
    name = step.get("name")
    klass = step.get("reaction_class")
    if name and len(name) <= _NAME_LIMIT:
        return name
    if klass and len(klass) <= _NAME_LIMIT:
        return klass
    if name:
        return name[:_NAME_LIMIT - 1].rstrip() + "\u2026"
    return "unnamed reaction"


def reaction_label(step, font=None, max_px=180, max_lines=3):
    """Wrapped caption for one retro step."""
    font = font or _font()
    lines = [f"Step {step['step']}"]
    lines.extend(_wrap_to_px(display_name(step), font, max_px, max_lines))
    prob = (step.get("metadata") or {}).get("policy_probability")
    if prob is not None:
        try:
            lines.append(format_probability(float(prob)))
        except (TypeError, ValueError):
            pass
    return lines


def format_probability(prob):
    """Render a policy probability without flattening small ones to 0.00.

    Two decimals turns 0.004 into "p = 0.00", which reads as though the model
    had no opinion rather than a confidently poor one. The low values are
    exactly the ones a student should notice.
    """
    if prob >= 0.01:
        return f"p = {prob:.2f}"
    if prob >= 0.001:
        return f"p = {prob:.3f}"
    return "p < 0.001"


def _label_colour(index, line):
    """Head line black, body blue, an unconfident probability amber."""
    if index == 0:
        return "black"
    if line.startswith("p "):
        try:
            low = "<" in line or float(line.split("=")[1]) < 0.05
        except (IndexError, ValueError):
            low = "<" in line
        if low:
            return "#B06A00"
    return "#1A4C7C"


# ==========================================================================
# Route drawing, with reaction names
# ==========================================================================
class LabeledRouteImageFactory:
    """Draw a route and write each reaction's name beside its node.

    Adapted from rxnutils.routes.image.RouteImageFactory. The original walks
    from a molecule straight into children[0]["children"] -- stepping over the
    reaction node and keeping nothing from it, which is why the stock route
    pictures carry no reaction information. Here the reaction node is captured
    on the way down and drawn on the way back up.
    """

    def __init__(self, route, in_stock_colors=None, show_all=True,
                 margin=210, mol_size=300, mol_draw_kwargs=None, font_size=15):
        in_stock_colors = in_stock_colors or {True: "green", False: "orange"}
        self.show_all = show_all
        self.margin = margin
        self._font = _font(font_size)
        self._font_bold = _font(font_size, bold=True)
        self._line_h = font_size + 3
        self._steps = []
        self._step_counter = 0

        self._stock_lookup, self._mol_lookup = {}, {}
        self._extract_molecules(route)
        images = molecules_to_images(
            list(self._mol_lookup.values()),
            [in_stock_colors[v] for v in self._stock_lookup.values()],
            size=mol_size, draw_kwargs=mol_draw_kwargs or {},
        )
        self._image_lookup = dict(zip(self._mol_lookup.keys(), images))

        self._mol_tree = self._extract_mol_tree(route)
        self._add_effective_size(self._mol_tree)
        pos0 = (
            self._mol_tree["eff_width"] - self._mol_tree["image"].width + self.margin,
            int(self._mol_tree["eff_height"] * 0.5)
            - int(self._mol_tree["image"].height * 0.5),
        )
        self._add_pos(self._mol_tree, pos0)

        self.image = Image.new(
            self._mol_tree["image"].mode,
            (self._mol_tree["eff_width"] + self.margin, self._mol_tree["eff_height"]),
            color="white")
        self._draw = ImageDraw.Draw(self.image)
        self._make_image(self._mol_tree)
        self.image = crop_image(self.image)

    # -- layout (unchanged from the original) -----------------------------
    def _extract_molecules(self, tree_dict):
        if tree_dict.get("type") == "mol":
            self._stock_lookup[tree_dict["smiles"]] = tree_dict.get("in_stock", False)
            self._mol_lookup[tree_dict["smiles"]] = Chem.MolFromSmiles(tree_dict["smiles"])
        for child in tree_dict.get("children", []):
            self._extract_molecules(child)

    def _add_effective_size(self, tree_dict):
        children = tree_dict.get("children", [])
        for child in children:
            self._add_effective_size(child)
        if children:
            tree_dict["eff_height"] = (sum(c["eff_height"] for c in children)
                                       + self.margin * (len(children) - 1))
            tree_dict["eff_width"] = (max(c["eff_width"] for c in children)
                                      + tree_dict["image"].size[0] + self.margin)
        else:
            tree_dict["eff_height"] = tree_dict["image"].size[1]
            tree_dict["eff_width"] = tree_dict["image"].size[0] + self.margin

    def _add_pos(self, tree_dict, pos):
        tree_dict["left"], tree_dict["top"] = pos[0], pos[1]
        children = tree_dict.get("children")
        if not children:
            return
        mid_y = pos[1] + int(tree_dict["image"].height * 0.5)
        children_height = (sum(c["eff_height"] for c in children)
                           + self.margin * (len(children) - 1))
        leftmost = min(pos[0] - self.margin - c["image"].width for c in children)
        child_y = mid_y - int(children_height * 0.5)
        child_ys = []
        for child in children:
            y_adjust = int((child["eff_height"] - child["image"].height) * 0.5)
            child_ys.append(child_y + y_adjust)
            child_y += self.margin + child["eff_height"]
        for idx, (child, child_y0) in enumerate(zip(children, child_ys)):
            y = child_y0
            if not child.get("children") and idx == 0 and len(children) > 1:
                y = child_ys[idx + 1] - self.margin - child["image"].height
            elif not child.get("children") and idx > 0:
                y = child_ys[idx - 1] + self.margin + children[idx - 1]["image"].height
            self._add_pos(child, (leftmost, y))

    # -- capture the reaction node ----------------------------------------
    def _extract_mol_tree(self, tree_dict):
        dict_ = {"smiles": tree_dict["smiles"],
                 "image": self._image_lookup[tree_dict["smiles"]]}
        children = tree_dict.get("children")
        if children:
            rxn_node = children[0]
            self._step_counter += 1
            step = {
                "step": self._step_counter,
                "product": tree_dict["smiles"],
                "reactants": [g["smiles"] for g in rxn_node.get("children", [])],
                "metadata": rxn_node.get("metadata") or {},
                "name": rxn_node.get("name"),
                "reaction_class": rxn_node.get("reaction_class"),
            }
            step["reaction_smiles"] = f"{'.'.join(step['reactants'])}>>{step['product']}"
            self._steps.append(step)
            dict_["step"] = step
            dict_["children"] = [
                self._extract_mol_tree(g) for g in rxn_node.get("children", [])
                if not (g.get("hide", False) and not self.show_all)
            ]
        return dict_

    # -- draw --------------------------------------------------------------
    def _make_image(self, tree_dict):
        self.image.paste(tree_dict["image"], (tree_dict["left"], tree_dict["top"]))
        children = tree_dict.get("children")
        if not children:
            return
        children_right = max(c["left"] + c["image"].width for c in children)
        mid_x = children_right + int(0.5 * (tree_dict["left"] - children_right))
        mid_y = tree_dict["top"] + int(tree_dict["image"].height * 0.5)

        self._draw.line((tree_dict["left"], mid_y, mid_x, mid_y), fill="black")
        for child in children:
            self._make_image(child)
            child_mid_y = child["top"] + int(0.5 * child["image"].height)
            self._draw.line((mid_x, mid_y, mid_x, child_mid_y,
                             child["left"] + child["image"].width, child_mid_y),
                            fill="black")
        self._draw.ellipse((mid_x - 8, mid_y - 8, mid_x + 8, mid_y + 8),
                           fill="black", outline="black")
        self._draw_label(tree_dict.get("step"), mid_x, mid_y,
                         tree_dict["left"] - children_right)

    def _draw_label(self, step, mid_x, mid_y, gap):
        if not step:
            return
        max_px = max(70, gap - 24)
        lines = reaction_label(step, font=self._font, max_px=max_px)
        if not lines:
            return
        widths = [_text_width(self._font_bold if i == 0 else self._font, ln)
                  for i, ln in enumerate(lines)]
        block_w, block_h = max(widths), self._line_h * len(lines)
        pad = 5
        top = mid_y - 14 - block_h
        if top < 2:
            top = mid_y + 14
        left = mid_x - block_w / 2.0
        # Opaque panel: without it the vertical connector runs through the text.
        self._draw.rectangle(
            (left - pad, top - pad, left + block_w + pad, top + block_h + pad - 3),
            fill="white", outline="#C8C8C8")
        for i, line in enumerate(lines):
            font = self._font_bold if i == 0 else self._font
            self._draw.text((mid_x - widths[i] / 2.0, top + i * self._line_h),
                            line, fill=_label_colour(i, line), font=font)

    @property
    def steps(self):
        return self._steps


def route_image(route_dict, margin=210, font_size=15, in_stock_colors=None):
    """Draw a route with reaction names, undecorated.

    Returns (PIL image, step records). Wrap with add_header_and_legend for the
    title band and colour key; the PDF wants only the key, since it prints its
    own heading.
    """
    factory = LabeledRouteImageFactory(
        route_dict, margin=margin, font_size=font_size,
        in_stock_colors=in_stock_colors or {True: "green", False: "orange"},
        show_all=True)
    return factory.image, factory.steps


def add_header_and_legend(img, title=None, subtitle=None, stock_label="in stock"):
    """Title above the scheme, colour key below it.

    The frame colours are the only thing telling a student whether a compound
    is purchasable, and nothing on the stock image says so.
    """
    font_t, font_s, font_l = _font(19, bold=True), _font(14), _font(13)
    head_h = (27 if title else 0) + (20 if subtitle else 0)
    if head_h:
        head_h += 8
    legend_h = 40

    out = Image.new("RGB", (max(img.width, 620), img.height + head_h + legend_h), "white")
    draw = ImageDraw.Draw(out)
    y = 6
    if title:
        draw.text((14, y), title, fill="black", font=font_t)
        y += 27
    if subtitle:
        draw.text((14, y), subtitle, fill="#555555", font=font_s)
    out.paste(img, (0, head_h))

    ly = img.height + head_h + 12
    draw.line((14, ly - 8, out.width - 14, ly - 8), fill="#DDDDDD")
    x = 14
    for colour, text in (("green", f"{stock_label} — buy this"),
                         ("orange", "not in stock — must be made")):
        draw.rounded_rectangle((x, ly + 1, x + 22, ly + 15), radius=5,
                               outline=colour, width=2)
        draw.text((x + 29, ly), text, fill="#333333", font=font_l)
        x += 29 + int(_text_width(font_l, text)) + 26
    draw.ellipse((x + 4, ly + 4, x + 14, ly + 14), fill="black")
    draw.text((x + 21, ly), "reaction; Step 1 is the last bond formed",
              fill="#333333", font=font_l)
    return out


# ==========================================================================
# ChemDraw-friendly export
# ==========================================================================
def _mol_with_coords(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    AllChem.Compute2DCoords(mol)   # else ChemDraw stacks every atom at the origin
    # Kekulize before writing: RDKit otherwise emits MDL bond type 4
    # (aromatic), which ChemDraw draws as a circle inside the ring instead of
    # alternating double bonds, so students redraw every ring.
    try:
        Chem.Kekulize(mol, clearAromaticFlags=True)
    except Exception:
        pass
    return mol


def step_rxn_block(step):
    """MDL rxn block for one retro step, written in the forward direction."""
    reactants = [_mol_with_coords(s) for s in step["reactants"]]
    product = _mol_with_coords(step["product"])
    if product is None or any(m is None for m in reactants):
        return None
    rxn = rdRxn.ChemicalReaction()
    for mol in reactants:
        rxn.AddReactantTemplate(mol)
    rxn.AddProductTemplate(product)
    return rdRxn.ReactionToRxnBlock(rxn)


def route_reaction_smiles(steps):
    out = ["# Reaction SMILES - one step per line, written forward.",
           "# ChemDraw: copy a line, then Edit > Paste Special > SMILES.",
           "# Listed in the order you would run them in the lab, so the step",
           "# numbers (which follow the retrosynthetic scheme) count down.",
           ""]
    for step in sorted(steps, key=lambda s: -s["step"]):
        out.append(f"# Step {step['step']}: {step.get('name') or 'unnamed reaction'}")
        out.append(step["reaction_smiles"])
        out.append("")
    return "\n".join(out)


def route_sdf(route_dict):
    """Every molecule in the route as a named, 2D SDF."""
    roles = {}

    def walk(node, depth=0):
        if node.get("type") == "mol" and node["smiles"] not in roles:
            roles[node["smiles"]] = {
                "depth": depth,
                "in_stock": bool(node.get("in_stock", False)),
                "sources": ", ".join(node.get("stock_sources") or []),
                "role": ("target" if depth == 0 else
                         ("starting material" if not node.get("children")
                          else "intermediate")),
            }
        for child in node.get("children", []):
            walk(child, depth + (1 if node.get("type") == "reaction" else 0))

    walk(route_dict)
    buf = io.StringIO()
    writer = Chem.SDWriter(buf)
    for i, (smiles, info) in enumerate(
            sorted(roles.items(), key=lambda kv: kv[1]["depth"]), 1):
        mol = _mol_with_coords(smiles)
        if mol is None:
            continue
        mol.SetProp("_Name", f"{i:02d}_{info['role'].replace(' ', '_')}")
        mol.SetProp("SMILES", smiles)
        mol.SetProp("role", info["role"])
        mol.SetProp("in_stock", "yes" if info["in_stock"] else "no")
        mol.SetProp("catalogue", info["sources"] or "-")
        mol.SetProp("depth", str(info["depth"]))
        writer.write(mol)
    writer.close()
    return buf.getvalue()


_CHEMDRAW_README = """How to open these files in ChemDraw
===================================

Set the page to landscape first: File > Page Setup > Landscape, then
View > Fit to Window. These schemes are laid out to fit a landscape page, and
on a portrait page ChemDraw pushes the right-hand structures off the edge and
wraps the reaction names into a narrow column.

To make that the default for every new document, open a blank document, set
Page Setup to Landscape, and save it as your stationery (File > Save As,
stationery/stylesheet format).

route.cdxml     ChemDraw's own format: the whole route, every step on its own
                row, earliest step at the bottom. Open it and you have real,
                editable structures laid out ready to arrange. Start here.

step_N.cdxml    The same thing one step at a time, reactants on the left and
                the product on the right. Useful when you only want one step.

step_N.rxn      MDL reaction files, one per step, written in the FORWARD
                direction (reactants left, product right).
                ChemDraw: File > Open. Unlike the CDXML files these carry
                reaction roles, so the arrow is already in place -- but MDL is
                a lossier format. Use these if you want the arrow for free.

reactions.smi   The same steps as reaction SMILES, one per line.
                ChemDraw: copy a line, then Edit > Paste Special > SMILES.
                Quicker, but you get no layout.

molecules.sdf   Every compound in the route with 2D coordinates, the role it
                plays, and which catalogue it was found in.
                ChemDraw: File > Open, or drag into a ChemDraw document.

route.png       The scheme as drawn in the notebook.

Step numbers follow the retrosynthetic scheme, so Step 1 is the LAST bond
formed in the lab. Writing the synthesis up, run them in reverse: highest
number first.

These routes are machine-generated predictions. They have not been checked
against the literature. Confirm precedent and conditions before attempting
any of them.
"""


def write_route_bundle(folder, route_dict, steps, image=None):
    """Write the ChemDraw-friendly files for one route."""
    os.makedirs(folder, exist_ok=True)
    written = []
    for step in steps:
        block = step_rxn_block(step)
        if block is not None:
            path = os.path.join(folder, f"step_{step['step']}.rxn")
            with open(path, "w") as fh:
                fh.write(block)
            written.append(path)
        cdxml = step_cdxml(step)
        if cdxml is not None:
            path = os.path.join(folder, f"step_{step['step']}.cdxml")
            with open(path, "w") as fh:
                fh.write(cdxml)
            written.append(path)

    whole = route_cdxml(route_dict, steps)
    if whole is not None:
        path = os.path.join(folder, "route.cdxml")
        with open(path, "w") as fh:
            fh.write(whole)
        written.append(path)

    for name, text in (("reactions.smi", route_reaction_smiles(steps)),
                       ("molecules.sdf", route_sdf(route_dict)),
                       ("HOW_TO_OPEN_IN_CHEMDRAW.txt", _CHEMDRAW_README)):
        if not text.strip():
            continue
        path = os.path.join(folder, name)
        with open(path, "w") as fh:
            fh.write(text)
        written.append(path)

    if image is not None:
        path = os.path.join(folder, "route.png")
        image.save(path)
        written.append(path)
    return written


def starting_materials(route_dict):
    """Leaf molecules of a route, with stock flag and catalogue."""
    out, seen = [], set()

    def walk(node):
        if node.get("type") == "mol" and not node.get("children"):
            if node["smiles"] not in seen:
                seen.add(node["smiles"])
                out.append({"smiles": node["smiles"],
                            "in_stock": bool(node.get("in_stock", False)),
                            "sources": node.get("stock_sources") or []})
        for child in node.get("children", []):
            walk(child)

    walk(route_dict)
    return out


# ==========================================================================
# PDF report
# ==========================================================================
# Only ZINC ships with the notebook, so it is the only label worth fixing.
# Any other catalogue is one the user added themselves, under a label they
# chose, and that label is printed as given.
SOURCE_LABELS = {
    "zinc": "ZINC (2020 snapshot)",
}


def _pretty_sources(sources):
    if not sources:
        return "not in stock"
    return ", ".join(SOURCE_LABELS.get(s, s) for s in sources)


def save_routes_pdf(path, routes, target_smiles, stats=None):
    """One scheme per page, each followed by its step and materials tables."""
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_LEFT
    from reportlab.lib.pagesizes import landscape, letter
    from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
    from reportlab.lib.units import inch
    from reportlab.platypus import (Image as RLImage, PageBreak, Paragraph,
                                    SimpleDocTemplate, Spacer, Table, TableStyle)

    page_w, page_h = landscape(letter)
    margin = 0.5 * inch
    avail_w = page_w - 2 * margin

    ss = getSampleStyleSheet()
    h1 = ParagraphStyle("h1", parent=ss["Heading1"], fontSize=15, spaceAfter=2,
                        textColor=colors.HexColor("#1A1A1A"))
    h2 = ParagraphStyle("h2", parent=ss["Heading2"], fontSize=11, spaceBefore=8,
                        spaceAfter=3, textColor=colors.HexColor("#1A4C7C"))
    small = ParagraphStyle("small", parent=ss["Normal"], fontSize=8,
                           textColor=colors.HexColor("#555555"), alignment=TA_LEFT)
    cell = ParagraphStyle("cell", parent=ss["Normal"], fontSize=7.5, leading=9.5)
    mono = ParagraphStyle("mono", parent=cell, fontName="Courier", fontSize=6.5,
                          leading=8.5)

    style_tbl = TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#DCE8E0")),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, 0), 7.5),
        ("GRID", (0, 0), (-1, -1), 0.4, colors.HexColor("#BBBBBB")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("LEFTPADDING", (0, 0), (-1, -1), 4),
        ("RIGHTPADDING", (0, 0), (-1, -1), 4),
        ("TOPPADDING", (0, 0), (-1, -1), 3),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
    ])

    story = [Paragraph("Retrosynthetic analysis", h1),
             Paragraph(f"Target: <font face='Courier'>{target_smiles}</font>",
                       ss["Normal"])]
    if stats:
        story.append(Paragraph(" &nbsp;|&nbsp; ".join(f"{k}: {v}" for k, v in stats.items()),
                               small))
    story.append(Spacer(1, 10))

    # Draw the target. A page headed by a SMILES string and nothing else asks
    # the reader to parse it in their head.
    target_png = os.path.join(os.path.dirname(os.path.abspath(path)),
                              "_target.png")
    try:
        mol = Chem.MolFromSmiles(target_smiles)
        if mol is not None:
            AllChem.Compute2DCoords(mol)
            Draw.MolToFile(mol, target_png, size=(520, 380))
            # MolToFile pads generously; trim it so the heading, the structure
            # and the note below sit together instead of across half a page.
            with Image.open(target_png) as im:
                crop_image(im, margin=6).save(target_png)
            with Image.open(target_png) as im:
                iw, ih = im.size
            height = 1.75 * inch
            flow = RLImage(target_png, width=iw * height / ih, height=height)
            flow.hAlign = "LEFT"
            story.append(flow)
            story.append(Spacer(1, 10))
    except Exception:
        pass
    story.append(Paragraph(
        "Routes generated with AiZynthFinder (Genheden et al., J. Cheminform. "
        "2020, 12, 70) using the public USPTO expansion policy. Reaction names "
        "assigned by Rxn-INSIGHT (Dobbelaere et al., J. Cheminform. 2024, 16, "
        "37); they group reactions by the bond formed rather than by mechanism, "
        "so a metal-free SNAr and a palladium-catalysed amination share a "
        "heading. Step numbers follow the retrosynthetic scheme, so Step 1 is "
        "the last bond formed. These are machine-generated predictions that "
        "have not been checked against the literature — confirm precedent "
        "and conditions before attempting any of them.", small))

    for i, route in enumerate(routes, 1):
        story.append(PageBreak())
        score = route.get("score")
        story.append(Paragraph(
            f"Route {i}" + (f" — score {score:.3f}" if score is not None else ""), h1))

        steps = route.get("steps") or []
        mats = route.get("starting_materials") or []

        # Build the tables first so their height is measured, not guessed;
        # a guess leaves the last row orphaned on a page of its own.
        tail = []
        if steps:
            rows = [["Step", "Reaction name", "Class", "p", "Reactants → product"]]
            for step in sorted(steps, key=lambda s: s["step"]):
                prob = (step.get("metadata") or {}).get("policy_probability")
                rows.append([
                    Paragraph(str(step["step"]), cell),
                    Paragraph(step.get("name") or "<i>unnamed</i>", cell),
                    Paragraph(step.get("reaction_class") or "—", cell),
                    Paragraph(format_probability(float(prob)).replace("p = ", "")
                              .replace("p ", "") if prob is not None else "—", cell),
                    Paragraph(step["reaction_smiles"].replace(">>", " &gt;&gt; "), mono),
                ])
            tbl = Table(rows, colWidths=[0.45 * inch, 1.85 * inch, 1.35 * inch,
                                         0.4 * inch, avail_w - 4.05 * inch],
                        repeatRows=1)
            tbl.setStyle(style_tbl)
            tail += [Paragraph("Steps", h2), tbl]

        if mats:
            rows = [["#", "SMILES", "Where to buy it"]]
            for n, mat in enumerate(mats, 1):
                rows.append([Paragraph(str(n), cell),
                             Paragraph(mat["smiles"], mono),
                             Paragraph(_pretty_sources(mat.get("sources")), cell)])
            tbl = Table(rows, colWidths=[0.4 * inch, avail_w - 2.9 * inch, 2.5 * inch],
                        repeatRows=1)
            tbl.setStyle(style_tbl)
            tail += [Paragraph("Starting materials", h2), tbl]

        tail_h = 0.0
        for flow in tail:
            try:
                tail_h += flow.wrap(avail_w, page_h)[1]
                # getSpaceBefore/After read the style; a plain getattr on a
                # Paragraph returns nothing and the headings' spacing is lost.
                tail_h += flow.getSpaceBefore() + flow.getSpaceAfter()
            except Exception:
                tail_h += 0.4 * inch

        img_path = route.get("image_path")
        if img_path and os.path.exists(img_path):
            with Image.open(img_path) as im:
                iw, ih = im.size
            max_h = max(1.6 * inch,
                        page_h - 2 * margin - 0.45 * inch - tail_h - 0.22 * inch)
            scale = min(avail_w / iw, max_h / ih, 1.0)
            flow = RLImage(img_path, width=iw * scale, height=ih * scale)
            flow.hAlign = "LEFT"
            story.append(flow)
            story.append(Spacer(1, 6))

        story.extend(tail)

    SimpleDocTemplate(path, pagesize=landscape(letter),
                      leftMargin=margin, rightMargin=margin,
                      topMargin=margin, bottomMargin=margin,
                      title="Retrosynthetic analysis",
                      author="Dr. Sunil Paliwal").build(story)
    return path


# ==========================================================================
# Annotating a route tree with reaction names
# ==========================================================================
def annotate_names(route_dict, cache=None, progress=None):
    """Attach a name and class to every reaction node in a route tree.

    A reaction node holds only its reactants; the product is the parent
    molecule, so the forward reaction SMILES has to be assembled while walking
    down. The cache matters: the same disconnection recurs across routes, and
    atom-mapping each one again is the slowest thing here.
    """
    cache = {} if cache is None else cache

    def walk(node, parent_smiles=None):
        if node.get("type") == "reaction":
            reactants = [c["smiles"] for c in node.get("children", [])
                         if c.get("type") == "mol"]
            rxn_smiles = f"{'.'.join(reactants)}>>{parent_smiles}"
            node["reaction_smiles"] = rxn_smiles
            if rxn_smiles not in cache:
                cache[rxn_smiles] = name_reaction(rxn_smiles)
                if progress:
                    progress(len(cache))
            node["name"], node["reaction_class"] = cache[rxn_smiles]
        for child in node.get("children", []):
            walk(child, node["smiles"] if node.get("type") == "mol" else parent_smiles)

    walk(route_dict)
    return cache


# ==========================================================================
# Downloads
# ==========================================================================
def download_button(path, label=None, primary=False):
    """An HTML download button that works in Colab.

    IPython's FileLink emits a bare filesystem path, which the browser
    resolves against localhost -- and in Colab the frontend is not served from
    the VM, so it refuses the connection. google.colab.files.download is also
    unreliable from inside a widget callback. Embedding the file as a data URI
    sidesteps both: the bytes travel with the HTML and the browser saves them
    without contacting anything.
    """
    import base64
    import html as _html
    import mimetypes

    html_escape = _html.escape

    name = os.path.basename(path)
    label = label or name
    mime = mimetypes.guess_type(name)[0] or "application/octet-stream"
    size_mb = os.path.getsize(path) / 1e6
    if size_mb > 25:
        # base64 adds a third again, and the whole thing is stored in the
        # notebook's output. Past this size, point at the file instead.
        return (f'<div style="font-family:sans-serif;font-size:13px;'
                f'margin:4px 0;">{html_escape(label)} is '
                f'{size_mb:,.1f} MB &mdash; too large to embed. Open it from the '
                f'folder icon in the left sidebar: <code>{html_escape(path)}</code></div>')
    with open(path, "rb") as fh:
        payload = base64.b64encode(fh.read()).decode()
    bg, fg, border = (("#1A4C7C", "#FFFFFF", "#1A4C7C") if primary
                      else ("#FFFFFF", "#1A4C7C", "#1A4C7C"))
    return (
        f'<a download="{_html.escape(name)}" href="data:{mime};base64,{payload}" '
        f'style="display:inline-block;margin:4px 10px 4px 0;padding:9px 18px;'
        f'background:{bg};color:{fg};border:1px solid {border};border-radius:6px;'
        f'font-family:sans-serif;font-size:13px;font-weight:600;'
        f'text-decoration:none;">{_html.escape(label)} '
        f'<span style="font-weight:400;opacity:.75;">({size_mb:,.1f} MB)</span></a>'
    )


def download_buttons(*specs):
    """Render several download buttons. Each spec is (path, label[, primary])."""
    from IPython.display import HTML, display
    html_parts = []
    for spec in specs:
        path, label = spec[0], spec[1]
        primary = spec[2] if len(spec) > 2 else False
        if path and os.path.exists(path):
            html_parts.append(download_button(path, label, primary))
    display(HTML("".join(html_parts)))



# ==========================================================================
# CDXML: ChemDraw's own format, with reaction arrows
# ==========================================================================
# A scheme without arrows is a pile of structures. CDXML records the arrow
# twice over: once as an <arrow> graphic, which is what gets drawn, and once
# as a <scheme>/<step> naming which fragments are reactants and which are
# products, which is what makes it a reaction rather than a picture of one.
# Both are written here, and RDKit's own CDXML reaction reader is used to
# check the result parses back to the reaction it came from.

# US Letter at 72 units per inch, less a half-inch margin on every side.
_PAGE_W, _PAGE_H, _PAGE_MARGIN = 612.0, 792.0, 54.0
_BOND_LENGTH = 30.0


def _layout_left_to_right(mols, gap=2.0, final_gap=None):
    """Shift each molecule so they sit side by side, in place.

    final_gap widens the space before the last molecule, leaving room for the
    arrow: the ordinary inter-molecule gap is only wide enough for a plus.
    """
    from rdkit.Geometry import Point3D

    x_cursor = 0.0
    for index, mol in enumerate(mols):
        if index == len(mols) - 1 and final_gap is not None:
            x_cursor += final_gap - gap
        conf = mol.GetConformer()
        xs = [conf.GetAtomPosition(i).x for i in range(mol.GetNumAtoms())]
        shift = x_cursor - min(xs)
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, Point3D(pos.x + shift, pos.y, pos.z))
        x_cursor += (max(xs) - min(xs)) + gap
    return mols


def _shift_y(mols, dy):
    from rdkit.Geometry import Point3D
    for mol in mols:
        conf = mol.GetConformer()
        for i in range(mol.GetNumAtoms()):
            pos = conf.GetAtomPosition(i)
            conf.SetAtomPosition(i, Point3D(pos.x, pos.y + dy, pos.z))


def _fragment_bounds(fragment):
    points = [tuple(map(float, n.attrib["p"].split()))
              for n in fragment.iter("n") if "p" in n.attrib]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]
    return min(xs), max(xs), min(ys), max(ys)


def _cdxml_text(text, x, y, size=10.0, bold=False):
    """A CDXML text object, centred horizontally on x."""
    import xml.etree.ElementTree as ET
    node = ET.Element("t", {"p": f"{x:.2f} {y:.2f}",
                            "BoundingBox": f"{x:.2f} {y - size:.2f} "
                                           f"{x + size:.2f} {y:.2f}",
                            "Justification": "Center"})
    style = {"size": f"{size:.1f}", "face": "1" if bold else "0"}
    ET.SubElement(node, "s", style).text = text
    return node


def _merge_cdxml(rows):
    """One CDXML document from rows of (reactant mols, product mol, label).

    Each molecule is written on its own and the fragments merged, rather than
    combining the molecules and writing once: RDKit would then emit a single
    <fragment> holding every atom, and ChemDraw treats that as one object, so
    selecting any structure selects all of them. Node ids are renumbered
    because bonds reference them by id and ids repeat across documents.
    """
    import xml.etree.ElementTree as ET
    from rdkit.Chem import rdmolfiles

    page = ET.Element("page", {"id": "2"})
    next_id, row_ids = 100, []

    for reactants, product, _label in rows:
        ids = []
        for mol in list(reactants) + [product]:
            root = ET.fromstring(rdmolfiles.MolToCDXMLBlock(mol))
            for fragment in root.iter("fragment"):
                remap = {}
                for node in fragment.iter():
                    if "id" in node.attrib:
                        remap[node.attrib["id"]] = str(next_id)
                        node.attrib["id"] = str(next_id)
                        next_id += 1
                for bond in fragment.iter("b"):
                    for end in ("B", "E"):
                        if end in bond.attrib and bond.attrib[end] in remap:
                            bond.attrib[end] = remap[bond.attrib[end]]
                fragment.set("id", str(next_id))
                next_id += 1
                page.append(fragment)
                ids.append(fragment.attrib["id"])
        row_ids.append(ids)

    # Arrows, placed from the measured gap between last reactant and product.
    by_id = {f.attrib["id"]: f for f in page.iter("fragment")}
    scheme = ET.Element("scheme", {"id": str(next_id)})
    next_id += 1
    arrows, texts = [], []
    for (_r, _p, label), ids in zip(rows, row_ids):
        if len(ids) < 2:
            continue
        last_reactant = _fragment_bounds(by_id[ids[-2]])
        prod = _fragment_bounds(by_id[ids[-1]])
        row = [_fragment_bounds(by_id[i]) for i in ids]
        y_mid = (min(b[2] for b in row) + max(b[3] for b in row)) / 2.0
        x1 = last_reactant[1] + 10.0
        x2 = prod[0] - 10.0
        if x2 - x1 < 18.0:                       # too tight to read as an arrow
            x1, x2 = last_reactant[1] + 4.0, prod[0] - 4.0

        arrow_id = str(next_id)
        next_id += 1
        arrows.append(ET.Element("arrow", {
            "id": arrow_id,
            "BoundingBox": f"{x1:.2f} {y_mid:.2f} {x2:.2f} {y_mid:.2f}",
            "FillType": "None", "ArrowheadHead": "Full",
            "ArrowheadType": "Solid", "HeadSize": "1000",
            "ArrowheadCenterSize": "875", "ArrowheadWidth": "250",
            "Tail3D": f"{x1:.2f} {y_mid:.2f} 0", "Head3D": f"{x2:.2f} {y_mid:.2f} 0",
        }))
        ET.SubElement(scheme, "step", {
            "id": str(next_id),
            "ReactionStepReactants": " ".join(ids[:-1]),
            "ReactionStepProducts": ids[-1],
            "ReactionStepArrows": arrow_id,
        })
        next_id += 1

        # Reaction name above the arrow, matching the scheme drawn in the
        # notebook: an arrow alone does not say what the reaction is.
        name_text, step_text = label if isinstance(label, tuple) else (label, None)
        cx = (x1 + x2) / 2.0
        caption_lines = wrap_caption(name_text)
        if caption_lines:
            size, line_h = 9.0, 11.0
            widest = max(len(line) for line in caption_lines)
            half = widest * size * _CHAR_W_FRACTION / 2.0
            # Bottom line sits just above the arrow; earlier lines stack up.
            y_bottom = y_mid - 11.0
            y_top = y_bottom - line_h * (len(caption_lines) - 1)
            for bx0, bx1, by0, by1 in row:
                if bx1 < cx - half or bx0 > cx + half:
                    continue
                if y_bottom + 2.0 >= by0 and y_top - size - 2.0 <= by1:
                    shift = y_top - (by0 - size - 3.0)
                    y_bottom -= shift
                    y_top -= shift
            for i, line in enumerate(caption_lines):
                node = _cdxml_text(line, cx, y_top + i * line_h, size=size)
                node.set("id", str(next_id)); next_id += 1
                texts.append(node)
        if step_text:
            node = _cdxml_text(step_text, cx, y_mid + 15.0, size=7.5)
            node.set("id", str(next_id)); next_id += 1
            texts.append(node)

        # Plus signs between consecutive reactants.
        for left, right in zip(ids[:-2], ids[1:-1]):
            lb, rb = _fragment_bounds(by_id[left]), _fragment_bounds(by_id[right])
            node = _cdxml_text("+", (lb[1] + rb[0]) / 2.0, y_mid + 4.0, size=13.0)
            node.set("id", str(next_id)); next_id += 1
            texts.append(node)

    for arrow in arrows:
        page.append(arrow)
    for node in texts:
        page.append(node)
    if len(scheme):
        page.append(scheme)

    bond_length = _fit_to_page(page)
    page_w, page_h = _page_size(page)
    page.set("BoundingBox", f"0 0 {page_w:.0f} {page_h:.0f}")
    page.set("Width", f"{page_w:.0f}")
    page.set("Height", f"{page_h:.0f}")
    doc = ET.Element("CDXML", {"id": "1", "BondLength": f"{bond_length:.2f}"})
    doc.append(page)
    return ('<?xml version="1.0" encoding="UTF-8" ?>\n'
            '<!DOCTYPE CDXML SYSTEM '
            '"https://static.chemistry.revvitycloud.com/cdxml/CDXML.dtd" >\n'
            + ET.tostring(doc, encoding="unicode"))


def _page_size(page, margin=None):
    """Page big enough for the drawing at its native bond length.

    A two-step route is about 690 units wide where portrait Letter offers 504,
    and the drawing cannot be shrunk to fit because ChemDraw rescales it back.
    So the document declares the page it needs: landscape Letter when that is
    enough, larger when it is not.
    """
    margin = _PAGE_MARGIN if margin is None else margin
    xs, ys = [], []
    for node in page.iter("n"):
        if "p" in node.attrib:
            px, py = map(float, node.attrib["p"].split())
            xs.append(px)
            ys.append(py)
    for node in page.iter("t"):
        if "p" not in node.attrib:
            continue
        tx_, ty_ = map(float, node.attrib["p"].split())
        label = node.find("s")
        text = (label.text or "") if label is not None else ""
        size = float(label.get("size", "9")) if label is not None else 9.0
        half = len(text) * size * _CHAR_W_FRACTION / 2.0
        xs.extend([tx_ - half, tx_ + half])
        ys.extend([ty_ - size, ty_ + size * 0.3])
    if not xs:
        return _PAGE_H, _PAGE_W            # landscape Letter
    return (max(_PAGE_H, max(xs) + margin),
            max(_PAGE_W, max(ys) + margin))


def _fit_to_page(page, usable_w=None, usable_h=None, margin=None):
    """Translate and, if needed, shrink the page so it fits on paper.

    At a fixed bond length a three-step route is wider than a Letter page, and
    a document whose content starts at x=0 sits on the page edge, which is why
    the leftmost structure came out clipped. Arrows are transformed with the
    structures so they stay between the right molecules.
    """
    margin = _PAGE_MARGIN if margin is None else margin
    usable_w = (_PAGE_W - 2 * margin) if usable_w is None else usable_w
    usable_h = (_PAGE_H - 2 * margin) if usable_h is None else usable_h

    nodes = [n for n in page.iter("n") if "p" in n.attrib]
    if not nodes:
        return _BOND_LENGTH
    points = [tuple(map(float, n.attrib["p"].split())) for n in nodes]
    xs = [p[0] for p in points]
    ys = [p[1] for p in points]

    # Captions stick out beyond the structures, so measure them too or the
    # page is fitted around the molecules and the text hangs off the edge.
    for node in page.iter("t"):
        if "p" not in node.attrib:
            continue
        tx_, ty_ = map(float, node.attrib["p"].split())
        label = (node.find("s").text or "") if node.find("s") is not None else ""
        size = float(node.find("s").get("size", "9")) if node.find("s") is not None else 9.0
        half = len(label) * size * _CHAR_W_FRACTION / 2.0
        xs.extend([tx_ - half, tx_ + half])
        ys.extend([ty_ - size, ty_ + size * 0.3])

    # Deliberately no scaling. ChemDraw renormalises a drawing so the declared
    # BondLength matches its own default, so shrinking the coordinates and
    # reducing BondLength to match gets undone on open -- and then some, which
    # is what pushed the whole scheme off to the right. Translate only, keep
    # the bond length at its native value, and keep the layout narrow enough
    # to fit by shortening captions rather than by scaling.
    scale = 1.0
    x0, y0 = min(xs), min(ys)

    def tx(px, py):
        return (px - x0) * scale + margin, (py - y0) * scale + margin

    for node, (px, py) in zip(nodes, points):
        nx, ny = tx(px, py)
        node.attrib["p"] = f"{nx:.2f} {ny:.2f}"

    for node in page.iter("t"):
        if "p" in node.attrib:
            tx_, ty_ = tx(*map(float, node.attrib["p"].split()))
            node.attrib["p"] = f"{tx_:.2f} {ty_:.2f}"
            node.attrib.pop("BoundingBox", None)

    for node in page.iter("s"):
        if "size" in node.attrib:
            node.attrib["size"] = f"{float(node.attrib['size']) * scale:.1f}"

    for arrow in page.iter("arrow"):
        ax1, ay1, ax2, ay2 = map(float, arrow.attrib["BoundingBox"].split())
        nx1, ny1 = tx(ax1, ay1)
        nx2, ny2 = tx(ax2, ay2)
        arrow.attrib["BoundingBox"] = f"{nx1:.2f} {ny1:.2f} {nx2:.2f} {ny2:.2f}"
        arrow.attrib["Tail3D"] = f"{nx1:.2f} {ny1:.2f} 0"
        arrow.attrib["Head3D"] = f"{nx2:.2f} {ny2:.2f} 0"

    return _BOND_LENGTH * scale


def _step_label(step):
    """Captions for one arrow: (name above, step number below).

    The name comes from display_name, so ChemDraw and the notebook scheme
    always show the same words. The step number goes below the arrow, where
    it costs the name no width.
    """
    return display_name(step), f"Step {step['step']}"


# RDKit's CDXML writer scales conformer coordinates by roughly this much, so
# a gap specified in Angstroms lands at gap * _CDXML_SCALE page units.
_CDXML_SCALE = 19.0
_LABEL_CHAR_W = 5.6      # width of one character at the 9pt label size,
                         # measured against a ChemDraw render; 4.6 was
                         # optimistic and captions overran the arrowhead.
_CHAR_W_FRACTION = 0.62  # same ratio, for text measured at other sizes


_CAPTION_ONE_LINE = 22   # captions up to this length stay on one line


def wrap_caption(text, max_lines=2):
    """Split a caption into at most max_lines balanced lines.

    Sizing the arrow gap to hold a 40-character caption on one line makes the
    row wider than a page, and ChemDraw then wraps the text itself into a
    narrow column at the page edge. Wrapping here instead -- the same thing
    the notebook scheme does -- keeps the gap, and the row, half the width.
    """
    words = (text or "").split()
    if not words:
        return []
    # Short captions fit the gap as they are; wrapping them wastes a line.
    if len(text) <= _CAPTION_ONE_LINE:
        return [text]
    target = max(1, len(text) // max_lines)
    lines, cur = [], ""
    for word in words:
        trial = f"{cur} {word}".strip()
        if cur and len(trial) > target and len(lines) < max_lines - 1:
            lines.append(cur)
            cur = word
        else:
            cur = trial
    if cur:
        lines.append(cur)
    return lines[:max_lines]


def _gap_for_label(label, minimum=5.0):
    """Arrow gap wide enough for the caption written above it.

    Measured from the longest wrapped line, not the whole caption.
    """
    lines = wrap_caption(label)
    if not lines:
        return minimum
    widest = max(len(line) for line in lines)
    needed = (widest * _LABEL_CHAR_W + 26.0) / _CDXML_SCALE
    return max(minimum, needed)


def _step_rows(step):
    mols = [_mol_with_coords(s) for s in step["reactants"]]
    product = _mol_with_coords(step["product"])
    if product is None or any(m is None for m in mols):
        return None
    return mols, product


def step_cdxml(step):
    """CDXML for one step: reactants, arrow, product.

    Returns None when this RDKit has no CDXML writer (added after 2023), so
    the bundle degrades to .rxn rather than failing.
    """
    from rdkit.Chem import rdmolfiles
    if not hasattr(rdmolfiles, "MolToCDXMLBlock"):
        return None
    row = _step_rows(step)
    if row is None:
        return None
    try:
        reactants, product = row
        label = _step_label(step)
        _layout_left_to_right(list(reactants) + [product],
                              final_gap=_gap_for_label(label[0]))
        return _merge_cdxml([(reactants, product, label)])
    except Exception:
        return None


def route_cdxml(route_dict, steps):
    """CDXML for a whole route: one step per row, earliest at the bottom."""
    from rdkit.Chem import rdmolfiles
    if not hasattr(rdmolfiles, "MolToCDXMLBlock"):
        return None

    rows, y_cursor = [], 0.0
    for step in sorted(steps, key=lambda s: -s["step"]):
        row = _step_rows(step)
        if row is None:
            continue
        reactants, product = row
        label = _step_label(step)
        mols = list(reactants) + [product]
        _layout_left_to_right(mols, final_gap=_gap_for_label(label[0]))

        ys = [mol.GetConformer().GetAtomPosition(i).y
              for mol in mols for i in range(mol.GetNumAtoms())]
        _shift_y(mols, y_cursor - max(ys))
        y_cursor = (y_cursor - max(ys)) + min(ys) - 3.0
        rows.append((reactants, product, label))

    if not rows:
        return None
    try:
        return _merge_cdxml(rows)
    except Exception:
        return None

## Cell 4 &mdash; Search

Type a SMILES and press **Run search**.

Reading the scheme: it runs right to left. The target is on the right, the
things you buy are on the left, and each black dot is a reaction with its name
beside it. A **green** frame means the compound was found in a catalogue;
**orange** means it has to be made. Step numbers follow the retrosynthetic
scheme, so Step 1 is the *last* bond formed in the lab.

**If a complex target comes back unsolved**, the four settings at the top of the
cell are the ones to change. `MAX_STEPS` is the longest route the search will
consider &mdash; AiZynthFinder's default of 6 is often too short for an elaborate
molecule. Raising it alone will not help, though: a deeper tree is a much
bigger tree, so raise `TIME_LIMIT` and `ITERATIONS` with it. Something like
`MAX_STEPS = 9`, `TIME_LIMIT = 600`, `ITERATIONS = 500` is a reasonable second
attempt.

Be sceptical of what comes back. Every step is an independent prediction, so
the chance that a route contains at least one implausible step grows with its
length &mdash; a nine-step route is not three times riskier than a three-step one,
it is considerably worse than that. Long routes are worth reading step by step
rather than trusting the score.

In [ ]:
# ---- Cell 4: search --------------------------------------------------------
import glob, json, os, re, shutil, subprocess, sys, textwrap, time, warnings, zipfile
from pathlib import Path
warnings.filterwarnings("ignore")

import ipywidgets as w
from IPython.display import display, clear_output

sys.path.insert(0, "/content")
import azf_post
import importlib; importlib.reload(azf_post)

# ---- search settings -------------------------------------------------------
# MAX_ROUTES is how many of the ranked routes to show and export. It has
# nothing to do with how long a route may be -- that is MAX_STEPS.
MAX_ROUTES = 10     # routes to display and export
MAX_STEPS  = 6      # longest route the search will consider (AiZynthFinder default: 6)
TIME_LIMIT = 120    # seconds the search may run      (default: 120)
ITERATIONS = 100    # tree-search iterations          (default: 100)

# Search worker. Runs inside the Python 3.11 environment and writes the route
# trees as JSON; everything else happens back in this kernel.
Path("/content/azf_worker.py").write_text('''
import json, sys
from aizynthfinder.aizynthfinder import AiZynthFinder
from aizynthfinder.chem import Molecule

cfg = json.loads(sys.argv[1])
smiles, outdir, max_routes = cfg["smiles"], cfg["outdir"], cfg["max_routes"]

finder = AiZynthFinder(configfile="/content/config.yml")
finder.config.search.max_transforms = cfg["max_steps"]
finder.config.search.time_limit = cfg["time_limit"]
finder.config.search.iteration_limit = cfg["iterations"]
finder.stock.select(cfg["stocks"])
finder.expansion_policy.select("uspto")
finder.filter_policy.select("uspto")
finder.target_smiles = smiles
finder.tree_search()
finder.build_routes()

def tag_stock(node):
    """Record which catalogues each molecule was found in, not just whether."""
    if node.get("type") == "mol":
        try:
            node["stock_sources"] = finder.stock.availability_list(
                Molecule(smiles=node["smiles"]))
        except Exception:
            node["stock_sources"] = []
    for child in node.get("children", []):
        tag_stock(child)

routes = []
for tree, scores in list(zip(finder.routes.reaction_trees, finder.routes.scores))[:max_routes]:
    dict_ = tree.to_dict()
    tag_stock(dict_)
    score = None
    if isinstance(scores, dict) and scores:
        score = float(list(scores.values())[0])
    routes.append({"score": score, "tree": dict_})

json.dump({"stats": finder.extract_statistics(), "routes": routes},
          open(f"{outdir}/routes.json", "w"), default=str)
''')

smiles_box = w.Text(value="CC(=O)Oc1ccccc1C(=O)O", description="Target SMILES",
                    layout=w.Layout(width="650px"),
                    style={"description_width": "110px"})
run_button = w.Button(description="Run search", button_style="primary")
output = w.Output()
OUT = "/content/azf_out"


def _bundle(routes, target):
    """PDF plus a zip of ChemDraw files, returned as paths."""
    pdf = f"{OUT}/retrosynthesis.pdf"
    azf_post.save_routes_pdf(pdf, routes, target)
    zip_path = f"{OUT}/chemdraw_bundle.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for folder in sorted(glob.glob(f"{OUT}/route_*/")):
            for path in sorted(glob.glob(folder + "*")):
                zf.write(path, os.path.relpath(path, OUT))
    return pdf, zip_path


def run_search(_):
    with output:
        clear_output()
        needed = ["/content/azf_env/bin/python", "/content/config.yml"]
        gone = [p for p in needed if not os.path.exists(p)]
        if gone:
            print("Setup is incomplete - missing:", ", ".join(gone))
            print("Run Cell 1, then Cell 2 (wait for 'Cell 2 complete.'), then this cell.")
            return

        target = smiles_box.value.strip()
        stocks = globals().get("STOCKS", ["zinc"])
        print(f"Searching for routes to {target} ...")
        shutil.rmtree(OUT, ignore_errors=True)
        os.makedirs(OUT)

        t0 = time.time()
        settings = {"smiles": target, "outdir": OUT, "stocks": stocks,
                    "max_routes": MAX_ROUTES, "max_steps": MAX_STEPS,
                    "time_limit": TIME_LIMIT, "iterations": ITERATIONS}
        result = subprocess.run(
            ["/content/azf_env/bin/python", "/content/azf_worker.py",
             json.dumps(settings)],
            capture_output=True, text=True, cwd="/content")
        if result.returncode != 0:
            clear_output()
            print(f"Search failed (exit code {result.returncode}).\n")
            if result.stdout.strip():
                print("--- stdout ---\n", result.stdout[-1500:])
            print("--- stderr ---\n", result.stderr[-3000:])
            return

        data = json.load(open(f"{OUT}/routes.json"))
        clear_output()
        print(f"Search finished in {time.time()-t0:.0f}s")
        for key, value in data["stats"].items():
            print(f"  {key}: {value}")
        if not data["routes"]:
            print("\nNo routes found. Try a simpler target, or check the SMILES.")
            return

        print("\nNaming reactions...")
        cache = {}
        for route in data["routes"]:
            azf_post.annotate_names(route["tree"], cache)
        named = sum(1 for n, _ in cache.values() if n)
        print(f"  {named} of {len(cache)} unique reactions named")

        clear_output()
        print(f"Search finished in {time.time()-t0:.0f}s   "
              f"({len(data['routes'])} routes, {named}/{len(cache)} reactions named)")

        prepared = []
        for i, route in enumerate(data["routes"], 1):
            tree = route["tree"]
            bare, steps = azf_post.route_image(tree)
            mats = azf_post.starting_materials(tree)
            bought = sum(1 for m in mats if m["in_stock"])
            score = route.get("score")
            title = f"Route {i}"
            if score is not None:
                title += f"  -  score {score:.3f}"
            title += f", {len(steps)} step{'s' if len(steps) != 1 else ''}"
            title += f", {bought}/{len(mats)} precursors in stock"
            disp = azf_post.add_header_and_legend(
                bare, title, f"Target: {tree['smiles']}")
            pdf_img = azf_post.add_header_and_legend(bare)

            folder = f"{OUT}/route_{i}"
            os.makedirs(folder, exist_ok=True)
            pdf_png = f"{OUT}/route_{i}_pdf.png"
            pdf_img.save(pdf_png)
            azf_post.write_route_bundle(folder, tree, steps, image=disp)

            prepared.append({"image_path": pdf_png, "steps": steps, "score": score,
                             "starting_materials": mats})
            display(disp)
            for mat in mats:
                print(f"    {mat['smiles']}  ->  "
                      f"{azf_post._pretty_sources(mat.get('sources'))}")
            print()

        pdf, zip_path = _bundle(prepared, target)
        print("Downloads:")
        azf_post.download_buttons(
            (pdf, "PDF report", True),
            (zip_path, "ChemDraw files", False))
        print("(Both are also in /content/azf_out - folder icon in the left sidebar.)")


run_button.on_click(run_search)
display(smiles_box, run_button, output)

## Reading the results

**Search statistics.** `is_solved` is true when at least one route reaches
purchasable starting materials for every branch. `number_of_solved_routes`
counts how many did.

**Score.** AiZynthFinder's state score combines how much of the route is in
stock with how short it is. It says nothing about whether the chemistry works.
A one-step route from two purchasable compounds scores near 1.00 however
implausible the step.

**p** is the expansion policy's confidence in that single disconnection &mdash; how
strongly the neural network, trained on reactions from US patents, expects this
template to apply to this molecule. It is a probability over templates, so
values are small by nature; anything above about 0.1 is a confident call.

Read it against the score, because the two measure different things and
routinely disagree. **A route can score 0.99 while resting on a step with
p &lt; 0.001** &mdash; that combination means "both precursors are purchasable and it
is only one step, but the model barely recognises the reaction."

What a low **p** does *not* mean is that the chemistry is wrong. It means the
transformation is rare in the patent corpus the policy learned from, which
catches nonsense but also flags sound chemistry that patents simply record less
often. Protecting-group manipulations are the clearest case: routine at the
bench, underrepresented in reactions extracted from patents. A route that
protects an acid as its benzyl ester, acylates elsewhere, then deprotects can
be entirely correct and still score p = 0.01 at every step.

So treat amber as *"check this one yourself"* rather than *"discard this"*. Any
probability below 0.05 is printed in amber for that reason, and deciding which
of the two it is &mdash; rare-but-right or rare-because-wrong &mdash; is the judgement
this notebook cannot make for you. It is the main thing worth arguing about in
class.

**Reaction names** come from Rxn-INSIGHT (Dobbelaere *et al.*, *J. Cheminform.*
**2024**, *16*, 37), which classifies a reaction from its atom-mapped reaction
centre. The names group reactions by the bond formed rather than by mechanism,
so an activated aryl fluoride reacting with an amine appears under the
N-arylation heading even though it is an SNAr needing no metal. A step that
cannot be placed is shown as *unnamed* rather than guessed at.

**Catalogues.** Each starting material lists where it was found. `ZINC (2020
snapshot)` means purchasable somewhere as of 2020 &mdash; not that it is available
now. Any other label is a catalogue you added yourself, under the name you gave
it, and means the compound was in that catalogue on the day you converted it.
Nothing listed means no catalogue had it, and the route needs justifying.

**Questions worth asking of any route this produces**

1. Which step would you actually attempt first, and why?
2. Is there a functional group here that will not survive one of these steps?
3. The model proposes a disconnection because similar reactions appear in the
   patent literature. Is your substrate similar enough for that to transfer?
4. Where a route scores well but uses an unnamed step, what is that step doing?

---

**What this notebook is**

AiZynthFinder is used **unmodified**, installed from PyPI at a pinned version.
Every route, score and disconnection comes from it; none of its code is altered
or redistributed here. What this notebook adds sits downstream of that output:

* packaging it to run in Colab, whose default Python it no longer supports
* reaction names, from Rxn-INSIGHT &mdash; the public expansion policy carries no
  reaction classifications of its own
* per-catalogue availability, so a starting material reports *where* it was
  found rather than a single in-stock flag
* a PDF report, and a ChemDraw export that opens as an editable reaction scheme

The one piece of third-party code that *is* modified is the route-drawing
layout, adapted from `rxnutils`; the changes are recorded in the header of the
support module, as its Apache 2.0 licence requires.

**Citing this work**

> Paliwal, S. *AI-Enabled Drug Discovery and Synthesis.* GitHub, 2026.

Please cite AiZynthFinder as well as this notebook &mdash; it does the retrosynthesis.

**Underlying tools**

* Genheden, S. *et al.* AiZynthFinder: a fast, robust and flexible open-source
  software for retrosynthetic planning. *J. Cheminform.* **2020**, *12*, 70.
  [doi:10.1186/s13321-020-00472-1](https://doi.org/10.1186/s13321-020-00472-1)
* Saigiridharan, L. *et al.* AiZynthFinder 4.0: developments based on learnings
  from 3 years of industrial application. *J. Cheminform.* **2024**, *16*, 57.
  [doi:10.1186/s13321-024-00860-x](https://doi.org/10.1186/s13321-024-00860-x)
* Dobbelaere, M. R. *et al.* Rxn-INSIGHT: fast chemical reaction analysis using
  bond-electron matrices. *J. Cheminform.* **2024**, *16*, 37.
  [doi:10.1186/s13321-024-00834-z](https://doi.org/10.1186/s13321-024-00834-z)

Route drawing adapted from `rxnutils` (reaction-utils, AstraZeneca, Apache 2.0).